In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report


In [2]:
# Dataset & Dataloader

from torch.utils.data import Dataset,DataLoader
import torchvision.transforms as transforms

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [4]:
train_set = CIFAR10(root = "./data",train=True,download=True,transform=transform)
test_set = CIFAR10(root ="./data",train=False,download=True,transform=transform)

100%|██████████| 170M/170M [00:12<00:00, 13.2MB/s]


In [5]:
class catdogdataset(Dataset):
  def __init__(self,dataset):
    self.dataset=dataset
    self.indices=[]

    for i,label in enumerate(dataset.targets):
      if label == 3 or label == 5:
        self.indices.append(i)

  def __len__(self):
    return len(self.indices)

  def __getitem__(self,idx):
    image,label = self.dataset[self.indices[idx]]
    label = 0 if label == 3 else 1
    return image,label

In [7]:
trainset = catdogdataset(train_set)
testset = catdogdataset(test_set)


trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

In [8]:
class CNN(nn.Module):
 def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
,
    )


    self.fc_layer=nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),
        nn.Linear(256,10)

    )

 def forward(self,x):
   x=self.conv_layers(x)
   x=x.view(x.size(0),-1)
   x=self.fc_layer(x)

   return x



model=CNN()

criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())


In [9]:
epochs=10
for epoch in range(epochs):
  epoch_training_loss=0.0

  for images,labels in trainloader:
    optimizer.zero_grad()

    output=model.forward(images)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()

    epoch_training_loss+=loss.item()

print(f"epoch={(epoch==1)/ (epochs)} & loss={(epoch_training_loss)/len(trainloader)}")


epoch=0.0 & loss=0.18980845472045765


In [18]:
# validation
model.eval()
running_val_loss=0.0
y_true = []
y_pred = []

with torch.no_grad():
  for images,labels in testloader:
    output = model.forward(images)
    _,predicted = torch.max(output,1)
    y_true.extend(labels.cpu().numpy())
    y_pred.extend(predicted.cpu().numpy())

#metrics
print("Accuracy:",accuracy_score(y_true,y_pred))
print("Precision:",precision_score(y_true,y_pred))
print("F1 Score:",f1_score(y_true,y_pred))
print("Recall:",recall_score(y_true,y_pred))

#Confusion Matrics
print("cm:",confusion_matrix(y_true,y_pred))

# Classification Report
print(classification_report(y_true,y_pred))

with torch.no_grad():
  for images,labels in testloader:
    outputs=model.forward(images)
    loss=criterion(outputs,labels)

    running_val_loss+=loss.item()

print(f"validation loss={running_val_loss/len(testloader)}")







Accuracy: 0.79
Precision: 0.7959183673469388
F1 Score: 0.7878787878787878
Recall: 0.78
cm: [[800 200]
 [220 780]]
              precision    recall  f1-score   support

           0       0.78      0.80      0.79      1000
           1       0.80      0.78      0.79      1000

    accuracy                           0.79      2000
   macro avg       0.79      0.79      0.79      2000
weighted avg       0.79      0.79      0.79      2000

validation loss=0.5111246956512332


In [19]:
# Evaluation of CNN

correct_labels=0
total_labels=0
model.eval()

with torch.no_grad():
  for images,labels in testloader:
    output=model.forward(images)
    _,predicted=torch.max(output,1)

    correct_labels+=(predicted==labels).sum().item()
    total_labels+=labels.size(0)


print(f"accuracy={correct_labels/total_labels*100}")

accuracy=79.0
